# Vectorless RAG with RAGNav (PDF)

This cookbook mirrors PageIndex’s **Vectorless RAG** idea, but implemented locally in RAGNav.

In this notebook, **vectorless** means:
- **No vector DB**
- **No precomputed embeddings** (we do *not* build the vector index)
- Retrieval uses **BM25 + structure expansion**, then an LLM answers from retrieved context

> If you want the hybrid (BM25 + embeddings) mode, use the other RAGNav examples.


## Step 0: Install + set keys

```bash
%pip install -q "ragnav[mistral,pdf]"
export MISTRAL_API_KEY="..."
```


In [1]:
from ragnav.env import load_env
from ragnav.ingest.pdf import PdfIngestOptions, ingest_pdf_bytes
from ragnav.llm.mistral import MistralClient
from ragnav.retrieval import RAGNavIndex, RAGNavRetriever

load_env()
llm = MistralClient()


## Step 1: Download a PDF (PageIndex-style arXiv flow)

```python
pdf_url = "https://arxiv.org/pdf/2507.13334.pdf"
```


In [2]:
import requests

pdf_url = "https://arxiv.org/pdf/2507.13334.pdf"
pdf_bytes = requests.get(pdf_url, timeout=60).content


/Users/irfanalidv/.pyenv/versions/3.12.1/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


## Step 2: Ingest + build a **vectorless** index

We ingest the PDF into **per-page blocks** with `anchors={"page": N}`.
Then we build a RAGNav index with `build_vectors=False`.


In [3]:
doc, blocks = ingest_pdf_bytes(
    pdf_bytes,
    name="2507.13334.pdf",
    metadata={"source": "arxiv", "url": pdf_url},
    opts=PdfIngestOptions(max_pages=25),
)

index = RAGNavIndex.build(
    documents=[doc],
    blocks=blocks,
    llm=llm,
    build_vectors=False,  # <-- vectorless mode
)
retriever = RAGNavRetriever(index=index, llm=llm)

print("pages_ingested:", len(blocks))


pages_ingested: 25


## Step 3: Vectorless retrieval + answer generation

We explicitly disable vector usage in retrieval (`use_vectors=False`).


In [4]:
import json

query = "What are the evaluation methods used in this paper?"

hits = retriever.retrieve_raw(
    query,
    max_blocks=6,
    k_final=6,
    expand_structure=True,
    use_vectors=False,
    k_vec=0,
    w_vec=0.0,
)

print(json.dumps(hits, indent=2)[:1500] + "\n...")


[
  {
    "doc_id": "pdf:2507.13334.pdf",
    "block_id": "pdf:2507.13334.pdf#p10",
    "title": null,
    "anchors": {
      "page": 10
    },
    "content": "Dimension\nPrompt Engineering\nContext Engineering\nModel\nC = prompt (static string)\nC = A(c1, c2, . . . , cn) (dynamic, structured assembly)\nTarget\narg maxprompt P\u03b8(Y|prompt)\nF \u2217= arg maxF E\u03c4\u223cT [Reward(P\u03b8(Y|CF(\u03c4)), Y\u2217\n\u03c4 )]\nComplexity\nManual or automated search over a string space.\nSystem-level optimization of F = {A, Retrieve, Select, . . . }.\nInformation\nInformation content is fixed within the prompt.\nAims to maximize task-relevant information under constraint |C| \u2264Lmax.\nState\nPrimarily stateless.\nInherently stateful, with explicit components for cmem and cstate.\nScalability\nBrittleness increases with length and complexity.\nManages complexity through modular composition.\nError Analysis\nManual inspection and iterative refinement.\nSystematic evaluation and debuggi

In [5]:
from ragnav.display import print_wrapped

context = "\n\n".join(h["content"] for h in hits[:6])
prompt = f"""Answer the question using ONLY the provided context.

Question: {query}

Context:
{context}
"""

answer = llm.chat(messages=[{"role": "user", "content": prompt}], temperature=0)
print_wrapped(answer)


The provided context does not explicitly list the evaluation methods used in the paper.
However, it mentions the following related to evaluation:

- **Systematic evaluation and debugging of individual context functions** (under "Error
Analysis" in Table 1 for Context Engineering).
- **Comprehensive evaluation frameworks to assess contextual comprehension** (in the
section on "Emerging Applications" under Multimodal Context).
- **Evaluation methodologies** are discussed as a key aspect of the survey, covering
benchmarks and methodologies for assessing component-level and system-level capabilities
(in the "Evaluation" subsection under "Related Work").

No specific evaluation methods (e.g., benchmarks, metrics, or protocols) are detailed in
the given context.


## Example output (will vary)

- You should see `pages_ingested: ...`
- Retrieval output is a JSON list with `doc_id`, `block_id`, and `anchors.page`
- The final answer will vary by model/version and the page cap
